In [ ]:
# =============================================================================
# SSD (Single Shot MultiBox Detector) 目标检测模型
# =============================================================================
# SSD是一种单阶段目标检测器，直接在单次前向传播中完成目标分类和定位
# 核心思想：在多尺度特征图上预设锚框(Anchor)，同时预测类别和边界框偏移
# 与Faster R-CNN等两阶段检测器相比，SSD速度更快，但精度略低

%matplotlib inline
import torch
import torchvision
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

# =============================================================================
# 步骤1：定义类别预测层（Classification Head）
# =============================================================================
# 为每个锚框预测类别（包括背景类）
# 输出通道数 = num_anchors * (num_classes + 1)
#   - num_anchors：每个空间位置的锚框数量
#   - num_classes + 1：类别数+背景类（索引0通常是背景）
def cls_predictor(num_inputs, num_anchors, num_classes):
    """类别预测层
    
    使用3x3卷积保持空间分辨率，每个位置输出num_anchors * (num_classes+1)个预测值
    """
    return nn.Conv2d(num_inputs, num_anchors * (num_classes + 1),
                     kernel_size=3, padding=1)

# =============================================================================
# 步骤2：定义边界框预测层（Bounding Box Regression Head）
# =============================================================================
# 为每个锚框预测4个偏移量：(delta_x, delta_y, delta_w, delta_h)
# 这些偏移量用于将预设锚框调整为更精确的目标边界框
def bbox_predictor(num_inputs, num_anchors):
    """边界框预测层
    
    输出通道数 = num_anchors * 4，每个锚框预测4个坐标偏移
    """
    return nn.Conv2d(num_inputs, num_anchors * 4, kernel_size=3, padding=1)


def forward(x, block):
    """辅助函数：前向传播一个模块"""
    return block(x)


# =============================================================================
# 步骤3：处理多尺度预测结果
# =============================================================================
def flatten_pred(pred):
    """展平预测结果
    
    输入: (batch_size, num_anchors*(num_classes+1), h, w) 类别预测
         或 (batch_size, num_anchors*4, h, w) 边界框预测
    处理: 将通道维度移到最后，然后展平为 (batch_size, h*w*num_anchors*depth)
    目的: 便于将不同尺度的预测结果拼接在一起
    """
    # permute(0, 2, 3, 1): 将通道维移到最后，变为(N, H, W, C)格式
    # flatten(start_dim=1): 从维度1开始展平，保持batch维度
    return torch.flatten(pred.permute(0, 2, 3, 1), start_dim=1)


def concat_preds(preds):
    """合并多尺度预测
    
    SSD在多个特征层上进行预测（不同分辨率），需要将各层的预测结果拼接
    preds: 各层预测结果列表
    返回: 拼接后的预测张量
    """
    return torch.cat([flatten_pred(p) for p in preds], dim=1)


# =============================================================================
# 步骤4：定义下采样块（Down-sampling Block）
# =============================================================================
# 用于构建特征金字塔，逐步减小特征图尺寸同时增加通道数
def down_sample_blk(in_channels, out_channels):
    """高和宽减半块
    
    结构: Conv -> BN -> ReLU -> Conv -> BN -> ReLU -> MaxPool
    效果: 空间尺寸减半(H/2, W/2)，通道数变为out_channels
    """
    blk = []
    for _ in range(2):
        # 3x3卷积，padding=1保持空间尺寸
        blk.append(nn.Conv2d(in_channels, out_channels,
                             kernel_size=3, padding=1))
        blk.append(nn.BatchNorm2d(out_channels))
        blk.append(nn.ReLU())
        in_channels = out_channels
    # 2x2最大池化，步幅=2，使空间尺寸减半
    blk.append(nn.MaxPool2d(2))
    return nn.Sequential(*blk)


# =============================================================================
# 步骤5：定义基础网络（Base Network）
# =============================================================================
# 用于提取图像的基础特征，由多个down_sample_blk堆叠而成
def base_net():
    """基础网络块
    
    输入: 3通道图像 (batch, 3, H, W)
    输出: 64通道特征图 (batch, 64, H/8, W/8)
    经过3个down_sample_blk，空间尺寸变为原来的1/8
    """
    blk = []
    num_filters = [3, 16, 32, 64]  # 逐层增加的通道数
    for i in range(len(num_filters) - 1):
        blk.append(down_sample_blk(num_filters[i], num_filters[i+1]))
    return nn.Sequential(*blk)

# 测试基础网络输出形状
# 输入: (2, 3, 256, 256)，输出应为 (2, 64, 32, 32)
forward(torch.zeros((2, 3, 256, 256)), base_net()).shape


# =============================================================================
# 步骤6：构建多尺度特征模块
# =============================================================================
def get_blk(i):
    """获取第i个模块
    
    TinySSD架构包含5个阶段：
    - i=0: 基础网络（提取初级特征）
    - i=1: 额外下采样块（通道64->128）
    - i=2,3: 更多下采样块（保持128通道）
    - i=4: 全局池化（得到1x1特征图）
    """
    if i == 0:
        blk = base_net()
    elif i == 1:
        blk = down_sample_blk(64, 128)
    elif i == 4:
        blk = nn.AdaptiveMaxPool2d((1,1))  # 全局池化，输出1x1
    else:
        blk = down_sample_blk(128, 128)
    return blk


def blk_forward(X, blk, size, ratio, cls_predictor, bbox_predictor):
    """模块前向传播
    
    参数:
        X: 输入特征图
        blk: 特征提取模块
        size: 当前层锚框的尺度（相对于图像的比例）
        ratio: 当前层锚框的高宽比
        cls_predictor: 类别预测层
        bbox_predictor: 边界框预测层
    返回:
        Y: 输出特征图（传给下一层）
        anchors: 当前层生成的锚框
        cls_preds: 类别预测
        bbox_preds: 边界框预测
    """
    Y = blk(X)
    # multibox_prior生成锚框，sizes和ratios定义锚框的尺度和形状
    anchors = d2l.multibox_prior(Y, sizes=size, ratios=ratio)
    cls_preds = cls_predictor(Y)
    bbox_preds = bbox_predictor(Y)
    return (Y, anchors, cls_preds, bbox_preds)


# =============================================================================
# 步骤7：定义TinySSD模型
# =============================================================================
class TinySSD(nn.Module):
    """TinySSD目标检测模型
    
    这是一个简化版SSD，包含5个检测层：
    - 第0层: 32x32特征图，检测小目标
    - 第1层: 16x16特征图
    - 第2层: 8x8特征图
    - 第3层: 4x4特征图
    - 第4层: 1x1特征图，检测大目标
    
    每层的感受野不同，适合检测不同尺寸的目标
    """
    def __init__(self, num_classes, **kwargs):
        super(TinySSD, self).__init__(**kwargs)
        self.num_classes = num_classes
        # 每层的输入通道数
        idx_to_in_channels = [64, 128, 128, 128, 128]
        for i in range(5):
            # 使用setattr动态创建模块属性
            # 例如: self.blk_0, self.cls_0, self.bbox_0 等
            setattr(self, f'blk_{i}', get_blk(i))
            setattr(self, f'cls_{i}', cls_predictor(idx_to_in_channels[i],
                                                    num_anchors, num_classes))
            setattr(self, f'bbox_{i}', bbox_predictor(idx_to_in_channels[i],
                                                      num_anchors))

    def forward(self, X):
        """前向传播
        
        返回:
            anchors: 所有层的锚框拼接，shape (1, num_all_anchors, 4)
            cls_preds: 类别预测，shape (batch, num_all_anchors, num_classes+1)
            bbox_preds: 边界框预测，shape (batch, num_all_anchors, 4)
        """
        anchors, cls_preds, bbox_preds = [None] * 5, [None] * 5, [None] * 5
        for i in range(5):
            # 使用getattr获取动态创建的模块
            X, anchors[i], cls_preds[i], bbox_preds[i] = blk_forward(
                X, getattr(self, f'blk_{i}'), sizes[i], ratios[i],
                getattr(self, f'cls_{i}'), getattr(self, f'bbox_{i}'))
        # 拼接所有层的锚框（第0维是batch维度，这里anchor的batch=1）
        anchors = torch.cat(anchors, dim=1)
        # 拼接并重塑类别预测
        cls_preds = concat_preds(cls_preds)
        # reshape: (batch, -1, num_classes+1)，-1自动计算锚框数量
        cls_preds = cls_preds.reshape(
            cls_preds.shape[0], -1, self.num_classes + 1)
        # 拼接边界框预测
        bbox_preds = concat_preds(bbox_preds)
        return anchors, cls_preds, bbox_preds


# =============================================================================
# 步骤8：配置锚框参数
# =============================================================================
# sizes: 每层的锚框尺度（相对于图像的短边比例）
# 从浅层到深层，尺度逐渐增大（小特征图检测大目标）
sizes = [[0.2, 0.272], [0.37, 0.447], [0.54, 0.619], [0.71, 0.79], [0.88, 0.961]]

# ratios: 每层的锚框高宽比 [1, 2, 0.5] 分别对应 正方形、高矩形、宽矩形
ratios = [[1, 2, 0.5]] * 5  # 5层使用相同的高宽比配置

# 每层的锚框数量 = len(sizes[i]) + len(ratios[i]) - 1
# 原理: sizes提供2个不同尺度的锚框，ratios提供3种形状（但1:1正方形已包含在sizes中，所以-1）
num_anchors = len(sizes[0]) + len(ratios[0]) - 1  # = 4个锚框/位置


# =============================================================================
# 步骤9：训练配置
# =============================================================================
batch_size = 32
train_iter, _ = d2l.load_data_bananas(batch_size)  # 加载香蕉检测数据集
device, net = d2l.try_gpu(), TinySSD(num_classes=1)
# SGD优化器，使用weight_decay（L2正则化）防止过拟合
trainer = torch.optim.SGD(net.parameters(), lr=0.2, weight_decay=5e-4)


# =============================================================================
# 步骤10：定义损失函数
# =============================================================================
# 类别损失：交叉熵损失，用于区分目标类别和背景
cls_loss = nn.CrossEntropyLoss(reduction='none')
# 边界框损失：L1损失（平滑L1的简化版），用于回归边界框坐标
# 只计算正样本的边界框损失（背景锚框不参与边界框回归）
bbox_loss = nn.L1Loss(reduction='none')


def calc_loss(cls_preds, cls_labels, bbox_preds, bbox_labels, bbox_masks):
    """计算总损失
    
    参数:
        cls_preds: (batch, num_anchors, num_classes+1) 类别预测
        cls_labels: (batch, num_anchors) 类别标签
        bbox_preds: (batch, num_anchors, 4) 边界框预测
        bbox_labels: (batch, num_anchors, 4) 边界框标签
        bbox_masks: (batch, num_anchors, 4) 掩码，只计算正样本的损失
    返回:
        cls + bbox: 类别损失和边界框损失之和（每个样本的平均）
    """
    batch_size, num_classes = cls_preds.shape[0], cls_preds.shape[2]
    # 类别损失：reshape后计算，然后恢复形状并求batch平均
    cls = cls_loss(cls_preds.reshape(-1, num_classes),
                   cls_labels.reshape(-1)).reshape(batch_size, -1).mean(dim=1)
    # 边界框损失：先乘mask过滤负样本，然后求平均
    bbox = bbox_loss(bbox_preds * bbox_masks,
                     bbox_labels * bbox_masks).mean(dim=1)
    return cls + bbox


def cls_eval(cls_preds, cls_labels):
    """计算类别预测正确的数量"""
    # argmax(dim=-1)获取预测的类别索引
    # type(cls_labels.dtype)确保类型一致便于比较
    return float((cls_preds.argmax(dim=-1).type(
        cls_labels.dtype) == cls_labels).sum())


def bbox_eval(bbox_preds, bbox_labels, bbox_masks):
    """计算边界框预测的绝对误差和"""
    return float((torch.abs((bbox_labels - bbox_preds) * bbox_masks)).sum())


# =============================================================================
# 步骤11：训练循环
# =============================================================================
num_epochs, timer = 20, d2l.Timer()
animator = d2l.Animator(xlabel='epoch', xlim=[1, num_epochs],
                        legend=['class error', 'bbox mae'])
net = net.to(device)

for epoch in range(num_epochs):
    # metric记录：[类别正确数, 类别总数, bbox误差和, bbox样本数]
    metric = d2l.Accumulator(4)
    net.train()
    for features, target in train_iter:
        timer.start()
        trainer.zero_grad()
        X, Y = features.to(device), target.to(device)
        
        # 1. 前向传播：生成锚框并预测类别和偏移量
        anchors, cls_preds, bbox_preds = net(X)
        
        # 2. 生成标注：根据真实边界框为锚框分配类别和偏移标签
        # multibox_target实现锚框与GT的匹配（IoU>阈值视为正样本）
        bbox_labels, bbox_masks, cls_labels = d2l.multibox_target(anchors, Y)
        
        # 3. 计算损失
        l = calc_loss(cls_preds, cls_labels, bbox_preds, bbox_labels,
                      bbox_masks)
        
        # 4. 反向传播和参数更新
        l.mean().backward()
        trainer.step()
        
        # 5. 记录指标
        metric.add(cls_eval(cls_preds, cls_labels), cls_labels.numel(),
                   bbox_eval(bbox_preds, bbox_labels, bbox_masks),
                   bbox_labels.numel())
    
    # 计算并显示本轮指标
    # cls_err: 类别错误率 = 1 - 正确率
    # bbox_mae: 边界框平均绝对误差
    cls_err, bbox_mae = 1 - metric[0] / metric[1], metric[2] / metric[3]
    animator.add(epoch + 1, (cls_err, bbox_mae))

print(f'class err {cls_err:.2e}, bbox mae {bbox_mae:.2e}')
print(f'{len(train_iter.dataset) / timer.stop():.1f} examples/sec on '
      f'{str(device)}')

In [ ]:
# =============================================================================
# SSD模型预测与可视化
# =============================================================================
# 本代码加载训练好的SSD模型，对测试图像进行目标检测，并可视化检测结果

# =============================================================================
# 步骤1：加载测试图像
# =============================================================================
# 读取香蕉检测数据集的测试图像
test_image_path = '../data/banana-detection/banana2.png'
# torchvision.io.read_image返回shape为(C, H, W)的uint8张量
X = torchvision.io.read_image(test_image_path).unsqueeze(0).float()
# unsqueeze(0)添加batch维度，shape变为(1, C, H, W)
# float()转换为浮点型，便于后续处理

# 将tensor转换回图像格式用于显示
# squeeze(0)移除batch维度 -> permute(1, 2, 0)将(C, H, W)转为(H, W, C)
# long()转回整数，因为图像像素值应为整数
img = X.squeeze(0).permute(1, 2, 0).long()


# =============================================================================
# 步骤2：定义预测函数
# =============================================================================
def predict(X):
    """对输入图像进行目标检测预测
    
    参数:
        X: 输入图像tensor，shape (1, C, H, W)
    返回:
        output: 过滤后的检测结果，shape (num_detections, 6)
                每行格式: [class_id, confidence, xmin, ymin, xmax, ymax]
                class_id=-1表示被抑制的检测框（NMS后）
    """
    # 设置为评估模式，禁用Dropout和BN的统计更新
    net.eval()
    
    # 前向传播，不计算梯度以节省内存
    with torch.no_grad():
        anchors, cls_preds, bbox_preds = net(X.to(device))
    
    # 对类别预测应用softmax得到概率分布
    # cls_preds: (batch, num_anchors, num_classes+1)
    # permute(0, 2, 1)后: (batch, num_classes+1, num_anchors)
    cls_probs = F.softmax(cls_preds, dim=2).permute(0, 2, 1)
    
    # multibox_detection执行NMS（非极大值抑制）过滤冗余检测框
    # 参数：类别概率、边界框偏移、锚框、nms_threshold、score_threshold
    output = d2l.multibox_detection(cls_probs, bbox_preds, anchors)
    
    # 过滤掉被NMS抑制的框（class_id=-1）
    # output[0]表示batch中第0个样本的所有检测结果
    idx = [i for i, row in enumerate(output[0]) if row[0] != -1]
    return output[0, idx]


# 执行预测
output = predict(X)


# =============================================================================
# 步骤3：定义可视化函数
# =============================================================================
def display(img, output, threshold):
    """显示检测结果
    
    参数:
        img: 原图tensor，shape (H, W, C)
        output: predict函数的输出，包含检测结果
        threshold: 置信度阈值，只显示高于该阈值的检测结果
    """
    # 设置图像显示尺寸
    d2l.set_figsize((5, 5))
    # 显示原图
    fig = d2l.plt.imshow(img)
    
    # 遍历所有检测结果
    for row in output:
        # row[1]是检测置信度（0-1之间的概率）
        score = float(row[1])
        
        # 跳过低置信度的检测
        if score < threshold:
            continue
        
        # 获取图像尺寸用于坐标反归一化
        # 模型输出的bbox坐标是归一化的（0-1之间），需要乘以图像尺寸得到像素坐标
        h, w = img.shape[0:2]
        
        # row[2:6]包含[xmin, ymin, xmax, ymax]四个归一化坐标
        # 乘以(w, h, w, h)转换为像素坐标
        bbox = [row[2:6] * torch.tensor((w, h, w, h), device=row.device)]
        
        # 在图像上绘制边界框
        # '%.2f' % score将置信度格式化为2位小数
        # 'w'表示边界框颜色为白色
        d2l.show_bboxes(fig.axes, bbox, '%.2f' % score, 'w')


# =============================================================================
# 步骤4：显示检测结果
# =============================================================================
# threshold=0.9表示只显示置信度>90%的检测结果
# 可以根据需要调整阈值，降低阈值会看到更多检测但可能有误检
display(img, output.cpu(), threshold=0.9)